In [1]:
# Significant similarity to audio descriptions. 

import numpy as np
import pandas as pd

scores = np.load('scores/S1/perceived_speech/gpt_layers_3_5_7_9/wheretheressmoke.npz', allow_pickle=True)
print(scores.files)
score_names = ['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
score_files = {}
for name in score_names:
    score_files[name] = scores[name].item()

print(score_files['window_zscores'])

['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
{('wheretheressmoke', 'WER'): array([-2.83584442e-15,  1.02416281e+00,  8.09638345e-01,  5.84936013e-01,
        8.90026857e-01,  1.32067636e-01,  3.46895762e-01,  9.52630069e-01,
        1.20995588e+00,  1.10318691e+00,  1.16697591e+00,  2.34543223e+00,
        2.98662997e+00,  2.42529003e+00,  2.83314068e+00,  2.28785370e+00,
        3.11138623e+00,  2.63438441e+00,  1.54642163e+00,  1.02442668e+00,
        2.11911637e+00,  8.65884132e-01,  8.92268598e-01,  2.17105433e+00,
        2.35985820e+00,  1.41437547e-01,  2.77797329e-01,  1.61639361e+00,
        1.24052255e+00,  8.05653536e-01,  5.86767048e-01,  2.22327362e-01,
        2.94218804e-01,  1.19442208e+00,  2.19063736e+00,  2.02801257e+00,
        1.65664667e+00,  3.10879129e+00,  2.67088519e+00,  3.48789000e+00,
        2.94184818e+00,  2.11995052e+00,  1.18982615e+00,  1.83643427e+00,
        7.76786630e-01,  1.95771420e+00,  3.64504070e+00,  3.38396058e+00,
 

In [2]:
window_zscores = {'subject': [], 'gpt_layers': [], 'WER':[],'BLEU':[], 'METEOR':[], 'BERT':[]}

for j in ["default", "gpt_layers_3_5_7_9"]:
    for subject in [1,2,3]:
        for task in ['wheretheressmoke']:
            scores = np.load(f'scores/S{subject}/perceived_speech/{j}/{task}.npz', allow_pickle=True)['window_zscores'].item()
            # print(scores['window_zscores'].item())
            window_zscores['subject'].append(subject)
            window_zscores['gpt_layers'].append(j)
            window_zscores['WER'].append(scores[(task, 'WER')])
            window_zscores['BLEU'].append(scores[(task, 'BLEU')])
            window_zscores['METEOR'].append(scores[(task, 'METEOR')])
            window_zscores['BERT'].append(scores[(task, 'BERT')])

window_zscores.keys()

dict_keys(['subject', 'gpt_layers', 'WER', 'BLEU', 'METEOR', 'BERT'])

In [3]:
results_df = pd.DataFrame(window_zscores)

print(len(results_df.loc[0, 'WER']))
print(len(results_df.loc[1, 'WER']))
print(len(results_df.loc[2, 'WER']))
print('=')
print(563*3)
m=1689

from scipy.stats import norm
#convert to p val
results_df['WER'] = results_df['WER'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1-norm.cdf(z) for z in l])
#sort
for i in range(3):
    for met in ['WER','BLEU','METEOR','BERT']:
        results_df.loc[i,met].sort()
# to q and check threshold
def p_to_q(p, i):
    return p*m/i
results_df['WER'] = results_df['WER'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])

# S1=np.array(results_df.loc[0, 'BERT'] + results_df.loc[1, 'BERT'] + results_df.loc[2, 'BERT']).mean()
# S2=np.array(results_df.loc[3, 'BERT'] + results_df.loc[4, 'BERT'] + results_df.loc[5, 'BERT']).mean()
# S3=np.array(results_df.loc[6, 'BERT'] + results_df.loc[7, 'BERT'] + results_df.loc[8, 'BERT']).mean()
# print(S1,S2,S3,'BERT')

results_df

563
563
563
=
1689


,subject,gpt_layers,WER,BLEU,METEOR,BERT
0,1,default,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,2,default,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,3,default,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,1,gpt_layers_3_5_7_9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,2,gpt_layers_3_5_7_9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5,3,gpt_layers_3_5_7_9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [4]:
results_df

,subject,gpt_layers,WER,BLEU,METEOR,BERT
0,1,default,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,2,default,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,3,default,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,1,gpt_layers_3_5_7_9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,2,gpt_layers_3_5_7_9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5,3,gpt_layers_3_5_7_9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [5]:
# S1=np.array(results_df.loc[0, 'BERT']).mean()
# S2=np.array(results_df.loc[1, 'BERT']).mean()
# S3=np.array(results_df.loc[2, 'BERT']).mean()
# to_file = pd.DataFrame({'subject':[1,2,3], 'significantly_decoded': [S1,S2,S3]})
tocsv = results_df.drop(columns=['WER', 'BLEU', 'METEOR'])

tocsv['BERT'] = tocsv['BERT'].apply(lambda x : x.count(1)/len(x))

tocsv.to_csv('perceived_speech_sig_decoded.csv', index=False)

tocsv

,subject,gpt_layers,BERT
0,1,default,0.619893
1,2,default,0.642984
2,3,default,0.808171
3,1,gpt_layers_3_5_7_9,0.618117
4,2,gpt_layers_3_5_7_9,0.539964
5,3,gpt_layers_3_5_7_9,0.758437


In [6]:
results = np.load('results/S1/perceived_speech/wheretheressmoke.npz', allow_pickle=True)
result_names = results.files
result_files={}
for name in result_names:
    result_files[name] = results[name]
result_files

FileNotFoundError: [Errno 2] No such file or directory: 'results/S1/perceived_speech/wheretheressmoke.npz'